# LeetCode #131 — Palindrome Partitioning: backtracking + bảng palindrome dựng trước

**Mức độ:** Medium · **Chủ đề:** String, Dynamic Programming, Backtracking

Skeleton trên LeetCode:

```python
class Solution:
    def partition(self, s: str) -> List[List[str]]:
```

Mục tiêu của notebook:

1. Backtracking "chọn vị trí cắt": mỗi tầng chọn đoạn tiếp theo `s[start:end]`.
2. Kiểm tra palindrome `O(1)` bằng bảng DP `is_pal[i][j]` dựng trước — thay vì `O(n)` mỗi lần.
3. Hiểu vì sao số kết quả có thể lên tới `2^(n−1)` (khi `s = "aaa…a"`).
4. Liên hệ với #132 (số lần cắt ít nhất) — từ liệt kê sang tối ưu.

# 1. Bài toán là gì?

Cho chuỗi `s`, chia `s` thành các đoạn sao cho **mỗi đoạn là palindrome** (đọc xuôi ngược như nhau). Trả về **mọi** cách chia.

**Constraints (theo đề):**

- `1 <= s.length <= 16`
- `s` chỉ gồm chữ cái tiếng Anh viết thường.

---

## Ví dụ

```text
"aab"  ->  [["a", "a", "b"], ["aa", "b"]]
"a"    ->  [["a"]]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quay lui (backtracking)
2. Chuỗi đối xứng (palindrome)
3. Quy hoạch động (Dynamic Programming — DP)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [1]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [2]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [3]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [4]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [5]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.2 Chuỗi đối xứng (palindrome)

### Định nghĩa và kiểm tra

Chuỗi **palindrome** đọc xuôi hay ngược đều như nhau: `"aba"`, `"abba"`, `"a"`, `""`. Kiểm tra:

- Cách ngắn: `s == s[::-1]` (`O(n)` thời gian và bộ nhớ).
- Hai con trỏ từ hai đầu tiến vào, so từng cặp (`O(1)` bộ nhớ).

In [6]:
def la_doi_xung(s, i=0, j=None):
    j = len(s) - 1 if j is None else j
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True

for w in ["aba", "abba", "abca", "a", ""]:
    print(f"{w!r:>7}: {la_doi_xung(w)} | s == s[::-1]: {w == w[::-1]}")

  'aba': True | s == s[::-1]: True
 'abba': True | s == s[::-1]: True
 'abca': False | s == s[::-1]: False
    'a': True | s == s[::-1]: True
     '': True | s == s[::-1]: True


### Nở từ tâm (expand around center)

Một palindrome đối xứng quanh **tâm**. Tâm là một ký tự (độ dài lẻ, `"aba"`) hoặc **khe giữa** hai ký tự (độ dài chẵn, `"abba"`). Chuỗi dài `n` có `2n − 1` tâm. Từ mỗi tâm, nở hai đầu ra chừng nào hai ký tự còn bằng nhau.

```text
s = "babad", tâm tại 'a' (i = 1):  b[a]b → "bab" ✓ → nở tiếp: hết bên trái → dừng
```

In [7]:
def no_tu_tam(s, l, r):
    while l >= 0 and r < len(s) and s[l] == s[r]:
        l -= 1; r += 1
    return s[l + 1:r]                       # đoạn palindrome dài nhất quanh tâm này

s_pal = "cbbdabadc"
tot_nhat = ""
for i in range(len(s_pal)):
    for l, r in [(i, i), (i, i + 1)]:       # tâm lẻ, tâm chẵn
        p = no_tu_tam(s_pal, l, r)
        if len(p) > len(tot_nhat):
            tot_nhat = p
print("palindrome dài nhất của", repr(s_pal), "là", repr(tot_nhat))

palindrome dài nhất của 'cbbdabadc' là 'dabad'


### Bảng palindrome cho mọi đoạn: `pal[i][j]`

Khi cần hỏi "đoạn `s[i..j]` có đối xứng không?" **nhiều lần**, dựng trước bảng `O(n²)`:

```text
pal[i][j] = (s[i] == s[j]) and (j - i < 2 or pal[i+1][j-1])
```

Ô `(i, j)` cần ô `(i+1, j−1)` → duyệt `i` **từ cuối về đầu**.

In [8]:
def bang_doi_xung(s):
    n = len(s)
    pal = [[False] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            pal[i][j] = s[i] == s[j] and (j - i < 2 or pal[i + 1][j - 1])
    return pal

pal_demo = bang_doi_xung("aab")
for i, row in enumerate(pal_demo):
    print(f"   i={i}:", ["T" if v else "." for v in row])
print("các đoạn đối xứng:", [("aab"[i:j + 1]) for i in range(3) for j in range(i, 3) if pal_demo[i][j]])

   i=0: ['T', 'T', '.']
   i=1: ['.', 'T', '.']
   i=2: ['.', '.', 'T']
các đoạn đối xứng: ['a', 'aa', 'a', 'b']


## 2.3 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [9]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [10]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.4 Áp dụng vào bài này

Liệt kê mọi cách cắt chuỗi thành các đoạn **đối xứng** bằng **quay lui**: mỗi tầng chọn điểm cắt tiếp theo, chỉ đi tiếp nếu đoạn vừa cắt là palindrome. Kiểm tra nhanh bằng **bảng palindrome** `pal[i][j]` dựng trước bằng **DP**.

# 3. Nền tảng

## 3.1 Kiểm tra palindrome

In [11]:
from typing import List
from functools import lru_cache
import itertools, random

def is_palindrome(t: str) -> bool:
    return t == t[::-1]

def is_palindrome_two_pointers(t: str) -> bool:
    i, j = 0, len(t) - 1
    while i < j:
        if t[i] != t[j]:
            return False
        i += 1; j -= 1
    return True

for t in ["", "a", "aa", "ab", "aba", "abba", "abca"]:
    print(f"{t!r:>7} {is_palindrome(t)} {is_palindrome_two_pointers(t)}")

     '' True True
    'a' True True
   'aa' True True
   'ab' False False
  'aba' True True
 'abba' True True
 'abca' False False


## 3.2 Một cách chia = một tập vị trí cắt

Chuỗi dài `n` có `n − 1` khe giữa các ký tự; mỗi khe cắt hoặc không → `2^(n−1)` cách chia. Brute force: thử mọi tập khe.

In [12]:
def partitions_brute(s: str) -> List[List[str]]:
    n, out = len(s), []
    for mask in range(1 << (n - 1)):
        parts, start = [], 0
        for k in range(n - 1):
            if mask >> k & 1:
                parts.append(s[start:k + 1]); start = k + 1
        parts.append(s[start:])
        if all(is_palindrome(p) for p in parts):
            out.append(parts)
    return out

print(partitions_brute("aab"))

[['aa', 'b'], ['a', 'a', 'b']]


# 4. Ý tưởng chính: backtracking theo vị trí bắt đầu

`backtrack(start)`: đã chia xong `s[:start]`. Thử mọi `end > start` sao cho `s[start:end]` là palindrome, đặt nó vào `path`, rồi đi tiếp từ `end`. Khi `start == n` → một cách chia hoàn chỉnh.

```text
backtrack(start):
    if start == n: lưu path[:]; return
    for end in start+1 .. n:
        if s[start:end] là palindrome:
            path.append(s[start:end]); backtrack(end); path.pop()
```

Cắt tỉa tự nhiên: đoạn nào không phải palindrome thì **không đi sâu**.

In [13]:
def partition_basic(s: str) -> List[List[str]]:
    result, path = [], []
    def backtrack(start):
        if start == len(s):
            result.append(path[:])
            return
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            if piece == piece[::-1]:
                path.append(piece)
                backtrack(end)
                path.pop()
    backtrack(0)
    return result

print(partition_basic("aab"))
print(partition_basic("abba"))

[['a', 'a', 'b'], ['aa', 'b']]
[['a', 'b', 'b', 'a'], ['a', 'bb', 'a'], ['abba']]


# 5. Mô phỏng

In [14]:
def trace_partition(s):
    result, path = [], []
    def backtrack(start, depth):
        ind = "  " * depth
        if start == len(s):
            print(f"{ind}=> lưu {path}")
            result.append(path[:])
            return
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            if piece == piece[::-1]:
                print(f"{ind}chọn {piece!r}")
                path.append(piece)
                backtrack(end, depth + 1)
                path.pop()
            else:
                print(f"{ind}bỏ {piece!r} (không phải palindrome)")
    backtrack(0, 0)
    return result

_ = trace_partition("aab")

chọn 'a'
  chọn 'a'
    chọn 'b'
      => lưu ['a', 'a', 'b']
  bỏ 'ab' (không phải palindrome)
chọn 'aa'
  chọn 'b'
    => lưu ['aa', 'b']
bỏ 'aab' (không phải palindrome)


# 6. Tối ưu: bảng `is_pal[i][j]` dựng trước

`piece == piece[::-1]` tốn `O(độ dài)` và được gọi rất nhiều lần cho cùng một đoạn. Dựng bảng một lần:

```text
is_pal[i][j] = (s[i] == s[j]) and (j - i < 2 or is_pal[i+1][j-1])
```

`is_pal[i][j]` phụ thuộc `is_pal[i+1][j−1]` (đoạn bên trong) → duyệt `i` **giảm dần**. Tổng `O(n²)`.

In [15]:
def build_is_pal(s: str):
    n = len(s)
    is_pal = [[False] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            if s[i] == s[j] and (j - i < 2 or is_pal[i + 1][j - 1]):
                is_pal[i][j] = True
    return is_pal

P = build_is_pal("aabba")
print("     " + "  ".join("aabba"))
for i, row in enumerate(P):
    print(f"  {'aabba'[i]}  " + "  ".join("T" if (x and j >= i) else ("." if j >= i else " ") for j, x in enumerate(row)))

     a  a  b  b  a
  a  T  T  .  .  .
  a     T  .  .  T
  b        T  T  .
  b           T  .
  a              T


In [16]:
def partition_dp(s: str) -> List[List[str]]:
    n = len(s)
    is_pal = build_is_pal(s)
    result, path = [], []
    def backtrack(start):
        if start == n:
            result.append(path[:])
            return
        for end in range(start, n):
            if is_pal[start][end]:
                path.append(s[start:end + 1])
                backtrack(end + 1)
                path.pop()
    backtrack(0)
    return result

print(partition_dp("aab"))

[['a', 'a', 'b'], ['aa', 'b']]


Chú ý chỉ số: trong bảng, `end` là chỉ số **bao gồm**, nên đoạn là `s[start:end + 1]` và đi tiếp từ `end + 1`.

# 7. Phiên bản đúng format LeetCode

In [17]:
class Solution:
    def partition(self, s: str) -> List[List[str]]:
        n = len(s)
        is_pal = [[False] * n for _ in range(n)]
        for i in range(n - 1, -1, -1):
            for j in range(i, n):
                if s[i] == s[j] and (j - i < 2 or is_pal[i + 1][j - 1]):
                    is_pal[i][j] = True

        result, path = [], []

        def backtrack(start: int) -> None:
            if start == n:
                result.append(path[:])
                return
            for end in range(start, n):
                if is_pal[start][end]:
                    path.append(s[start:end + 1])
                    backtrack(end + 1)
                    path.pop()

        backtrack(0)
        return result

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS  {actual if len(str(actual)) < 70 else str(actual)[:67] + '...'}")


sol = Solution()
check("Ví dụ 1", sorted(sol.partition("aab")), [["a", "a", "b"], ["aa", "b"]])
check("Ví dụ 2", sol.partition("a"), [["a"]])
check("abba", sorted(sol.partition("abba")), [["a", "b", "b", "a"], ["a", "bb", "a"], ["abba"]])
check("Toàn 'a' (16)", len(sol.partition("a" * 16)), 2 ** 15)

random.seed(131)
for _ in range(2000):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 10)))
    expected = sorted(partitions_brute(s))
    assert sorted(sol.partition(s)) == expected == sorted(partition_basic(s))
print("2000 chuỗi ngẫu nhiên: khớp brute force qua mọi tập vị trí cắt")

Ví dụ 1        PASS  [['a', 'a', 'b'], ['aa', 'b']]
Ví dụ 2        PASS  [['a']]
abba           PASS  [['a', 'b', 'b', 'a'], ['a', 'bb', 'a'], ['abba']]
Toàn 'a' (16)  PASS  32768


2000 chuỗi ngẫu nhiên: khớp brute force qua mọi tập vị trí cắt


# 8. Vì sao đúng? Độ phức tạp?

- Mỗi cách chia hợp lệ tương ứng **duy nhất** một dãy đoạn `s[0:e1], s[e1:e2], …` — backtracking duyệt đúng các dãy đó (mỗi lần chọn đoạn tiếp theo bắt đầu tại `start`).
- Chỉ đi tiếp khi đoạn là palindrome → mọi kết quả đều hợp lệ; không bỏ sót vì thử **mọi** `end`.

```text
Dựng bảng:   O(n²)
Backtracking: O(n · 2^n) xấu nhất (s = "aaa…a" có 2^(n-1) kết quả, mỗi kết quả sao chép O(n))
Space:       O(n²) bảng + O(n) đệ quy (không tính output)
```

Với `n ≤ 16`, `2^15 = 32 768` kết quả — chấp nhận được.

# 9. Những lỗi dễ mắc

## Lỗi 1: Lệch chỉ số giữa bảng (bao gồm) và lát cắt (loại trừ)

In [19]:
def wrong_index(s):
    n = len(s)
    is_pal = build_is_pal(s)
    result, path = [], []
    def backtrack(start):
        if start == n:
            result.append(path[:]); return
        for end in range(start, n):
            if is_pal[start][end]:
                path.append(s[start:end])             # SAI: phải là end + 1
                backtrack(end + 1)
                path.pop()
    backtrack(0)
    return result

print("Sai :", wrong_index("aab"))

Sai : [['', '', ''], ['a', '']]


## Lỗi 2: Dựng bảng với `i` tăng dần

`is_pal[i+1][j−1]` chưa được tính khi `i` tăng → mọi palindrome dài ≥ 3 bị đánh giá sai.

In [20]:
def wrong_table_order(s):
    n = len(s)
    is_pal = [[False] * n for _ in range(n)]
    for i in range(n):                                # SAI: phải giảm dần
        for j in range(i, n):
            if s[i] == s[j] and (j - i < 2 or is_pal[i + 1][j - 1]):
                is_pal[i][j] = True
    return is_pal

print("Sai : 'aba' là palindrome?", wrong_table_order("aba")[0][2], "| Đúng:", build_is_pal("aba")[0][2])

Sai : 'aba' là palindrome? False | Đúng: True


## Lỗi 3: Lưu `path` thay vì `path[:]`

Mọi kết quả trỏ về cùng một list, cuối cùng đều rỗng.

## Lỗi 4: Tham lam lấy palindrome **dài nhất** ở mỗi bước

Chỉ cho **một** cách chia (và với bài #132 cũng không cho số lần cắt ít nhất).

In [21]:
def greedy_longest(s):
    out, i = [], 0
    while i < len(s):
        for j in range(len(s), i, -1):
            if is_palindrome(s[i:j]):
                out.append(s[i:j]); i = j; break
    return out

print("Tham lam:", greedy_longest("aab"), "— chỉ là một trong", len(sol.partition("aab")), "cách")
print("Tham lam cho 'abaab':", greedy_longest("abaab"), "| ít đoạn nhất có thể:", min(sol.partition("abaab"), key=len))

Tham lam: ['aa', 'b'] — chỉ là một trong 2 cách
Tham lam cho 'abaab': ['aba', 'a', 'b'] | ít đoạn nhất có thể: ['a', 'baab']


# 10. Mẹo nhận diện pattern

- **"Chia chuỗi thành các đoạn thỏa điều kiện, liệt kê mọi cách"** → backtracking theo vị trí bắt đầu (như #93 Restore IP).
- **Điều kiện kiểm tra lặp lại trên các đoạn con** → dựng bảng DP trước (`is_pal`).
- **Chỉ cần số lượng / giá trị tối ưu** → bỏ liệt kê, dùng DP trên tiền tố (#132, #139).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Liệt kê mọi cách chia `"aabb"` và `"racecar"`.

## Bài 2 — Điền chỗ trống (dựng bảng)

In [22]:
def exercise_2_build_is_pal(s: str):
    n = len(s)
    is_pal = [[False] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            # TODO
            pass
    return is_pal


# assert exercise_2_build_is_pal("aba")[0][2] is True
# assert exercise_2_build_is_pal("abca")[0][3] is False
# assert exercise_2_build_is_pal("abba")[0][3] is True and exercise_2_build_is_pal("abba")[1][2] is True

## Bài 3 — Biến thể: đếm số cách chia (không liệt kê)

Đếm số cách chia `s` thành các palindrome bằng DP trên tiền tố: `ways[i]` = số cách chia `s[:i]`.

```python
count_partitions("aab")      -> 2
count_partitions("a" * 16)   -> 32768
```

In [23]:
def exercise_3_count_partitions(s: str) -> int:
    # TODO
    pass


# assert exercise_3_count_partitions("aab") == 2
# assert exercise_3_count_partitions("a" * 16) == 32768
# assert exercise_3_count_partitions("ab") == 1

## Bài 4 — Biến thể: Palindrome Partitioning II (LeetCode #132, Hard)

Số lần **cắt ít nhất** để mọi đoạn là palindrome.

```python
min_cut("aab")  -> 1
min_cut("a")    -> 0
min_cut("ab")   -> 1
```

In [24]:
def exercise_4_min_cut(s: str) -> int:
    # TODO
    pass


# assert exercise_4_min_cut("aab") == 1
# assert exercise_4_min_cut("a") == 0
# assert exercise_4_min_cut("ab") == 1
# assert exercise_4_min_cut("abaab") == 1

## Bài 5 — Mini challenge: Palindromic Substrings (LeetCode #647)

Đếm số **đoạn con** (vị trí khác nhau thì tính khác nhau) là palindrome, `O(n²)` thời gian, `O(1)` bộ nhớ — bằng cách **nở từ tâm**.

```python
count_substrings("abc") -> 3
count_substrings("aaa") -> 6
```

In [25]:
def exercise_5_count_substrings(s: str) -> int:
    # TODO
    pass


# assert exercise_5_count_substrings("abc") == 3
# assert exercise_5_count_substrings("aaa") == 6
# assert exercise_5_count_substrings("abba") == 6

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_partition` sau khi tự làm.

**Bài 3**
- Mức 1: `ways[0] = 1`; `ways[j+1] += ways[i]` với mọi `i ≤ j` có `is_pal[i][j]`.

**Bài 4**
- Mức 1: `cut[j]` = số cắt ít nhất cho `s[:j+1]`; nếu `is_pal[0][j]` thì `0`.
- Mức 2: ngược lại `min(cut[i−1] + 1)` với mọi `i ≥ 1` có `is_pal[i][j]`.

**Bài 5**
- Mức 1: có `2n − 1` tâm (ký tự và khe); nở `l, r` khi `s[l] == s[r]`, mỗi lần nở đếm thêm 1.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [26]:
for w in ["aabb", "racecar"]:
    print(w, "->", sol.partition(w))

aabb -> [['a', 'a', 'b', 'b'], ['a', 'a', 'bb'], ['aa', 'b', 'b'], ['aa', 'bb']]
racecar -> [['r', 'a', 'c', 'e', 'c', 'a', 'r'], ['r', 'a', 'cec', 'a', 'r'], ['r', 'aceca', 'r'], ['racecar']]


## Đáp án Bài 2

In [27]:
def answer_2_build_is_pal(s: str):
    return build_is_pal(s)

assert answer_2_build_is_pal("aba")[0][2] is True
assert answer_2_build_is_pal("abca")[0][3] is False
assert answer_2_build_is_pal("abba")[0][3] is True and answer_2_build_is_pal("abba")[1][2] is True
print("Bài 2 PASS (giống build_is_pal ở mục 6)")

Bài 2 PASS (giống build_is_pal ở mục 6)


## Đáp án Bài 3

In [28]:
def answer_3_count_partitions(s: str) -> int:
    n = len(s)
    is_pal = build_is_pal(s)
    ways = [1] + [0] * n
    for j in range(n):
        for i in range(j + 1):
            if is_pal[i][j]:
                ways[j + 1] += ways[i]
    return ways[n]

assert answer_3_count_partitions("aab") == 2
assert answer_3_count_partitions("a" * 16) == 32768
assert answer_3_count_partitions("ab") == 1
for _ in range(1000):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 10)))
    assert answer_3_count_partitions(s) == len(sol.partition(s))
print("Bài 3 PASS")

Bài 3 PASS

## Đáp án Bài 4

In [29]:
def answer_4_min_cut(s: str) -> int:
    n = len(s)
    is_pal = build_is_pal(s)
    cut = [0] * n
    for j in range(n):
        if is_pal[0][j]:
            cut[j] = 0
        else:
            cut[j] = min(cut[i - 1] + 1 for i in range(1, j + 1) if is_pal[i][j])
    return cut[-1]

assert answer_4_min_cut("aab") == 1
assert answer_4_min_cut("a") == 0
assert answer_4_min_cut("ab") == 1
assert answer_4_min_cut("abaab") == 1
for _ in range(1000):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 10)))
    assert answer_4_min_cut(s) == min(len(p) for p in sol.partition(s)) - 1
print("Bài 4 PASS")

Bài 4 PASS


`min(...)` trong nhánh `else` luôn có ít nhất một phần tử: `is_pal[j][j]` (đoạn một ký tự) luôn đúng.

## Đáp án Bài 5

In [30]:
def answer_5_count_substrings(s: str) -> int:
    n, total = len(s), 0
    for center in range(2 * n - 1):
        l, r = center // 2, center // 2 + center % 2
        while l >= 0 and r < n and s[l] == s[r]:
            total += 1
            l -= 1; r += 1
    return total

assert answer_5_count_substrings("abc") == 3
assert answer_5_count_substrings("aaa") == 6
assert answer_5_count_substrings("abba") == 6
for _ in range(1000):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 12)))
    P = build_is_pal(s)
    assert answer_5_count_substrings(s) == sum(P[i][j] for i in range(len(s)) for j in range(i, len(s)))
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Backtracking chọn đoạn tiếp theo `s[start:end+1]` khi nó là palindrome; dựng trước bảng `is_pal[i][j] = s[i]==s[j] and (j−i<2 or is_pal[i+1][j−1])` với `i` giảm dần để kiểm tra `O(1)`. Đếm / tối ưu thì chuyển sang DP tiền tố.

---

## Checklist trước khi nộp

- [ ] Bảng `is_pal` duyệt `i` **giảm dần**?
- [ ] Chỉ số bao gồm/loại trừ khớp (`s[start:end+1]`, đi tiếp từ `end+1`)?
- [ ] Lưu bản sao `path[:]`?
- [ ] Không dùng tham lam?